# POLAPP YAMNet 음향 사건 모델 내보내기

TensorFlow Hub의 공식 YAMNet(AudioSet 521 클래스)을 ONNX로 변환합니다. 재학습 노트북이 아니며, POLAPP은 군중/중첩 발화와 비명·고함·충격·파손·총성 계열 점수만 의사결정 보조 근거로 사용합니다.

In [ ]:
!pip -q install 'tensorflow-hub>=0.16' 'tf2onnx>=1.16' 'onnxruntime>=1.19'

In [ ]:
import json, shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tf2onnx

ROOT = Path('/content/polapp_yamnet')
EXPORT = ROOT / 'yamnet'
EXPORT.mkdir(parents=True, exist_ok=True)
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

In [ ]:
class YamnetScores(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[tf.TensorSpec([None], tf.float32, name='waveform')])
    def __call__(self, waveform):
        scores, _, _ = self.model(waveform)
        return {'scores': scores}

wrapper = YamnetScores(yamnet)
signature = [tf.TensorSpec([None], tf.float32, name='waveform')]
tf2onnx.convert.from_function(
    wrapper.__call__, input_signature=signature, opset=17,
    output_path=str(EXPORT / 'yamnet.onnx')
)

In [ ]:
class_map_source = Path(yamnet.class_map_path().numpy().decode('utf-8'))
shutil.copy2(class_map_source, EXPORT / 'yamnet_class_map.csv')
groups = {
    'crowd': ['crowd', 'hubbub', 'speech babble', 'children playing'],
    'danger': ['screaming', 'shout', 'yell', 'glass', 'breaking', 'smash, crash', 'slap, smack', 'whack, thwack', 'thump, thud', 'gunshot', 'gunfire', 'explosion'],
}
(EXPORT / 'event_groups.json').write_text(json.dumps(groups, indent=2), encoding='utf-8')
(EXPORT / 'model_card.md').write_text('''# POLAPP YAMNet ONNX

Source: TensorFlow Hub google/yamnet/1
Training ontology: AudioSet 521 classes
Input: mono 16 kHz float32 waveform
Output: frame-level class scores

POLAPP uses selected scores as supporting evidence only. AudioSet is not a Korean police-field validation set; false alarms per hour and event F1 must be measured with a separate consented field set before operational use.
''', encoding='utf-8')

In [ ]:
# 변환 산출물 로드와 출력 차원을 확인합니다.
import onnxruntime as ort
session = ort.InferenceSession(str(EXPORT / 'yamnet.onnx'), providers=['CPUExecutionProvider'])
waveform = np.zeros(16000, dtype=np.float32)
input_meta = session.get_inputs()[0]
scores = session.run(None, {input_meta.name: waveform})[0]
assert scores.ndim == 2 and scores.shape[1] == 521, scores.shape
print('YAMNet ONNX output:', scores.shape)
archive_path = shutil.make_archive('/content/yamnet', 'zip', EXPORT)
from google.colab import files
files.download(archive_path)